# FMLib LightGBM + SHAP: campaign-compatible run

Notebook запускает FMLib с параметрами классического campaign-отбора из `2_2_feature_selection_s.ipynb`: пять стратифицированных fold, `seed=42`, `max_depth=5`, `n_estimators=500`, `learning_rate=0.05` и порог `0.85`.

Перед запуском замените `feature_drop.path` в `lightgbm_campaign_matched.yaml` на свой файл. Для сопоставимости с campaign передавайте уже подготовленный numeric train dataset; target и service-колонки не должны быть кандидатами.

In [ ]:
from pathlib import Path

REPO_ROOT = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'pyproject.toml').is_file()
)
CONFIG_PATH = REPO_ROOT / 'examples/configs/feature_selection/lightgbm_campaign_matched.yaml'

# CHANGE_ME: путь к уже подготовленному train parquet.
TRAIN_PATH = 'CHANGE_ME/path/to/train.parquet'
TARGET_COLUMN = 'target'
# Например: ('epk_id', 'report_dt'). Эти колонки не участвуют в отборе.
ID_COLUMNS: tuple[str, ...] = ()
TIME_COLUMN: str | None = None
# None = все числовые колонки, кроме target и service-колонок.
# Чтобы повторить campaign в точности, задайте здесь тот же список x.columns.
CONTINUOUS_COLUMNS: tuple[str, ...] | None = None
CATEGORICAL_COLUMNS: tuple[str, ...] = ()
OUTPUT_DIR = REPO_ROOT / 'artifacts/lgbm_campaign_matched'

if 'CHANGE_ME' in TRAIN_PATH:
    raise ValueError('Set TRAIN_PATH before running the notebook.')
if 'CHANGE_ME' in CONFIG_PATH.read_text(encoding='utf-8'):
    raise ValueError('Set feature_drop.path in lightgbm_campaign_matched.yaml before running.')

In [ ]:
from pyspark.sql import SparkSession

from fmlib.feature_selection import (
    FeatureSchema,
    FeatureSelectionConfig,
    FeatureSelectionPipeline,
)

spark = SparkSession.builder.getOrCreate()
config = FeatureSelectionConfig.from_yaml(CONFIG_PATH)
train_df = spark.read.parquet(TRAIN_PATH)
print(f'Loaded {train_df.count():,} train rows and {len(train_df.columns):,} columns.')

In [ ]:
service_columns = {TARGET_COLUMN, *ID_COLUMNS}
if TIME_COLUMN is not None:
    service_columns.add(TIME_COLUMN)

if CONTINUOUS_COLUMNS is None:
    numeric_prefixes = ('tinyint', 'smallint', 'int', 'bigint', 'float', 'double', 'decimal')
    continuous = tuple(
        name for name, dtype in train_df.dtypes
        if name not in service_columns and dtype.lower().startswith(numeric_prefixes)
    )
else:
    continuous = CONTINUOUS_COLUMNS

schema = FeatureSchema(
    categorical=CATEGORICAL_COLUMNS,
    continuous=continuous,
    target=TARGET_COLUMN,
    task_type='binary_classification',
    time=TIME_COLUMN,
    id_columns=ID_COLUMNS,
)
print(f'LightGBM/SHAP candidates: {len(schema.continuous):,}')
if not schema.continuous:
    raise ValueError('No continuous candidates. Set CONTINUOUS_COLUMNS explicitly.')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
result = FeatureSelectionPipeline(config).fit_select(
    spark,
    datasets={'train': train_df},
    schema=schema,
    output_dir=OUTPUT_DIR,
)

selected = list(result.selected_features)
print(f'Selected {len(selected):,} features.')
print(f'Result: {OUTPUT_DIR / "final_results.json"}')
selected[:20]

In [ ]:
lightgbm_scores = next(
    value for key, value in result.scores.items()
    if key.startswith('lightgbm#')
)
feature_drop_scores = next(
    value for key, value in result.scores.items()
    if key.startswith('feature_drop#')
)
print('Per-fold seeds:', lightgbm_scores['fold_seeds'])
print('Feature-drop report:', feature_drop_scores)

import pandas as pd
pd.DataFrame({
    'feature': list(lightgbm_scores['importances']),
    'lgbm_importance': list(lightgbm_scores['importances'].values()),
    'shap_importance': list(lightgbm_scores['shap_importances'].values()),
}).sort_values('shap_importance', ascending=False).head(20)